# 🩻 Health Data Extraction

Using [**Garmin Connect API**](https://github.com/cyberjunky/python-garminconnect)

From Garmin watch

In [ ]:
# TODO: Gather solar data with timestamp ranges per day. Will need to store in a separate dataset and link through `Date`.

In [ ]:
import re
import os
import sys
import importlib
import datetime
import requests
import getpass
from pathlib import Path
from tqdm import tqdm

import pandas as pd
import numpy as np

from garth.exc import GarthException, GarthHTTPError
from garminconnect import (
    Garmin,
    GarminConnectAuthenticationError,
    GarminConnectConnectionError,
    GarminConnectTooManyRequestsError,
)


# This is a test
import lib

importlib.reload(lib)

project = lib.Project()
log = lib.getLogger(project.name)

DATE_FORMAT = '%Y-%m-%d'


## ⌚️ Garmin helper functions to interact with API
Taken from example file provided in documentation to ensure safe usage with API

In [2]:
def safe_api_call(api_method, *args, **kwargs):
    """
    Safe API call wrapper with comprehensive error handling.

    This demonstrates the error handling patterns used throughout the library.
    Returns (success: bool, result: Any, error_message: str)
    """
    try:
        result = api_method(*args, **kwargs)
        return True, result, None

    except GarthHTTPError as e:
        # Handle specific HTTP errors gracefully
        error_str = str(e)
        status_code = getattr(getattr(e, "response", None), "status_code", None)

        if status_code == 400 or "400" in error_str:
            return (
                False,
                None,
                "Endpoint not available (400 Bad Request) - Feature may not be enabled for your account",
            )
        elif status_code == 401 or "401" in error_str:
            return (
                False,
                None,
                "Authentication required (401 Unauthorized) - Please re-authenticate",
            )
        elif status_code == 403 or "403" in error_str:
            return (
                False,
                None,
                "Access denied (403 Forbidden) - Account may not have permission",
            )
        elif status_code == 404 or "404" in error_str:
            return (
                False,
                None,
                "Endpoint not found (404) - Feature may have been moved or removed",
            )
        elif status_code == 429 or "429" in error_str:
            return (
                False,
                None,
                "Rate limit exceeded (429) - Please wait before making more requests",
            )
        elif status_code == 500 or "500" in error_str:
            return (
                False,
                None,
                "Server error (500) - Garmin's servers are experiencing issues",
            )
        elif status_code == 503 or "503" in error_str:
            return (
                False,
                None,
                "Service unavailable (503) - Garmin's servers are temporarily unavailable",
            )
        else:
            return False, None, f"HTTP error: {e}"

    except FileNotFoundError:
        return (
            False,
            None,
            "No valid tokens found. Please login with your email/password to create new tokens.",
        )

    except GarminConnectAuthenticationError as e:
        return False, None, f"Authentication issue: {e}"

    except GarminConnectConnectionError as e:
        return False, None, f"Connection issue: {e}"

    except GarminConnectTooManyRequestsError as e:
        return False, None, f"Rate limit exceeded: {e}"

    except Exception as e:
        return False, None, f"Unexpected error: {e}"

def get_credentials():
    """Get email and password from environment or user input."""
    email = os.getenv("EMAIL")
    password = os.getenv("PASSWORD")

    if not email:
        email = input("Login email: ")
    if not password:
        password = getpass("Enter password: ")

    return email, password

def init_api() -> Garmin | None:
    """Initialize Garmin API with authentication and token management."""

    # Configure token storage
    tokenstore = os.getenv("GARMINTOKENS", "~/.garminconnect")
    tokenstore_path = Path(tokenstore).expanduser()

    print(f"🔐 Token storage: {tokenstore_path}")

    # Check if token files exist
    if tokenstore_path.exists():
        print("📄 Found existing token directory")
        token_files = list(tokenstore_path.glob("*.json"))
        if token_files:
            print(
                f"🔑 Found {len(token_files)} token file(s): {[f.name for f in token_files]}"
            )
        else:
            print("⚠️ Token directory exists but no token files found")
    else:
        print("📭 No existing token directory found")

    # First try to login with stored tokens
    try:
        print("🔄 Attempting to use saved authentication tokens...")
        garmin = Garmin()
        garmin.login(str(tokenstore_path))
        print("✅ Successfully logged in using saved tokens!")
        return garmin

    except (
        FileNotFoundError,
        GarthHTTPError,
        GarminConnectAuthenticationError,
        GarminConnectConnectionError,
    ):
        print("🔑 No valid tokens found. Requesting fresh login credentials.")

    # Loop for credential entry with retry on auth failure
    while True:
        try:
            # Get credentials
            email, password = get_credentials()

            print("� Logging in with credentials...")
            garmin = Garmin(
                email=email, password=password, is_cn=False, return_on_mfa=True
            )
            result1, result2 = garmin.login()

            if result1 == "needs_mfa":
                print("🔐 Multi-factor authentication required")

                mfa_code = input("Please enter your MFA code: ")
                print("🔄 Submitting MFA code...")

                try:
                    garmin.resume_login(result2, mfa_code)
                    print("✅ MFA authentication successful!")

                except GarthHTTPError as garth_error:
                    # Handle specific HTTP errors from MFA
                    error_str = str(garth_error)
                    if "429" in error_str and "Too Many Requests" in error_str:
                        print("❌ Too many MFA attempts")
                        print("💡 Please wait 30 minutes before trying again")
                        sys.exit(1)
                    elif "401" in error_str or "403" in error_str:
                        print("❌ Invalid MFA code")
                        print("💡 Please verify your MFA code and try again")
                        continue
                    else:
                        # Other HTTP errors - don't retry
                        print(f"❌ MFA authentication failed: {garth_error}")
                        sys.exit(1)

                except GarthException as garth_error:
                    print(f"❌ MFA authentication failed: {garth_error}")
                    print("💡 Please verify your MFA code and try again")
                    continue

            # Save tokens for future use
            garmin.garth.dump(str(tokenstore_path))
            print(f"💾 Authentication tokens saved to: {tokenstore_path}")
            print("✅ Login successful!")
            return garmin

        except GarminConnectAuthenticationError:
            print("❌ Authentication failed:")
            print("💡 Please check your username and password and try again")
            # Continue the loop to retry
            continue

        except (
            FileNotFoundError,
            GarthHTTPError,
            GarminConnectConnectionError,
            requests.exceptions.HTTPError,
        ) as err:
            print(f"❌ Connection error: {err}")
            print("💡 Please check your internet connection and try again")
            return None

        except KeyboardInterrupt:
            print("\n👋 Cancelled by user")
            return None


### Get Garmin client

In [3]:
api = init_api()

🔐 Token storage: /Users/christophermagno/.garminconnect
📄 Found existing token directory
🔑 Found 2 token file(s): ['oauth2_token.json', 'oauth1_token.json']
🔄 Attempting to use saved authentication tokens...
✅ Successfully logged in using saved tokens!


## Helper functions for datetime

In [4]:
def _get_date_string(date):
    return date.strftime(DATE_FORMAT)

def today():
    return datetime.date.today().strftime(DATE_FORMAT)

def convert_epoch_to_datetime(epoch):
    return datetime.datetime.fromtimestamp(epoch / 1000)

def get_date_range(start=None, rng=None):
    start = start or datetime.datetime.today()
    rng = rng or int(start.strftime('%j'))
    dates = reversed([start - datetime.timedelta(days=x) for x in range(rng)])
    return [x.strftime(DATE_FORMAT) for x in dates]


## Helper functions to gather and organize Garmin data

Some useful methods from the Garmin class to use
* get_stats - using
* get_heart_rates - using
* get_sleep_data - using
* get_fitnessage_data - using
* get_activities - using
*
* get_steps_data
* get_daily_steps
* get_floors
* get_stress_data
* get_rhr_day
* get_hrv_data

Others to look at
* get_activities_fordate
* get_earned_badges

In [31]:
def get_device():
    """
    deviceId
    imageUrl
    productDisplayName
    deviceTypeSimpleName
    """
    devices = safe_api_call(api.get_devices)[1]
    return devices[0]

def get_sleep_data(date):

    sleep_data = {}

    to_pop = [
        'id',
        'userProfilePK',
        'napTimeSeconds',
        'sleepWindowConfirmed',
        'sleepWindowConfirmationType',
        'autoSleepStartTimestampGMT',
        'autoSleepEndTimestampGMT',
        'sleepQualityTypePK',
        'sleepResultTypePK',
        'deviceRemCapable',
        'retro',
        'sleepFromDevice',
        'sleepScores',
        'sleepScoreInsight',
        'sleepScorePersonalizedInsight',
        'sleepVersion',
        'averageSPO2',
        'averageSpO2HR',
        'lowestSPO2'
    ]

    data = safe_api_call(api.get_sleep_data, date)[1]

    sleep_data.update(data['dailySleepDTO'])
    if 'sleepScores' in sleep_data:
        sleep_data['sleepScore'] = sleep_data['sleepScores']['overall']['value']
        sleep_data['sleepScoreQuality'] = sleep_data['sleepScores']['overall']['qualifierKey']
        sleep_data['stressSleepQuality'] = sleep_data['sleepScores']['stress']['qualifierKey']
        sleep_data['awakeCountQuality'] = sleep_data['sleepScores']['awakeCount']['qualifierKey']
        sleep_data['remSleepQuality'] = sleep_data['sleepScores']['remPercentage']['qualifierKey']
        sleep_data['restlessnessSleepQuality'] = sleep_data['sleepScores']['restlessness']['qualifierKey']
        sleep_data['lightSleepQuality'] = sleep_data['sleepScores']['lightPercentage']['qualifierKey']
        sleep_data['deepSleepQuality'] = sleep_data['sleepScores']['deepPercentage']['qualifierKey']
    else:
        sleep_data['sleepScore'] = None
        sleep_data['sleepScoreQuality'] = None
        sleep_data['stressSleepQuality'] = None
        sleep_data['awakeCountQuality'] = None
        sleep_data['remSleepQuality'] = None
        sleep_data['restlessnessSleepQuality'] = None
        sleep_data['lightSleepQuality'] = None
        sleep_data['deepSleepQuality'] = None

    # result['sleepHeartRate'] = data['sleepHeartRate']
    sleep_data['avgOvernightHrv'] = data.get('avgOvernightHrv')
    sleep_data['hrvStatus'] = data.get('hrvStatus')
    sleep_data['restingHeartRate'] = data.get('restingHeartRate')

    # Convert timestamp to datetime
    for item in ['sleepStartTimestampGMT', 'sleepEndTimestampGMT', 'sleepStartTimestampLocal', 'sleepEndTimestampLocal']:
        sleep_data[item] = convert_epoch_to_datetime(sleep_data[item])

    for key in to_pop:
        try:
            sleep_data.pop(key)
        except KeyError as e:
            pass

    return sleep_data

def get_hydration_data(date):
    data = safe_api_call(api.get_hydration_data, date)[1]
    hydration_data = {
        'hydrationValueInML': data['valueInML'],
        'hydrationGoalInML': data['goalInML'],
        'sweatLossInML': data['sweatLossInML']
    }
    return hydration_data

def get_solar_data():
    """
    Example output
    {
    'localConnectDate': '2025-12-28',
     'userProfilePk': 126748136,
     'deviceId': 3476417250,
     'solarInputReadings': [{'readingTimestampLocal': '2025-12-28T00:00:00.0',
       'readingTimestampGmt': '2025-12-28T08:00:00.0',
       'solarUtilization': 0.0,
       'notChargingTooHot': False,
       'notChargingTooCold': False,
       'notChargingBatteryFull': False,
       'notChargingExternalPower': False,
       'notChargingUserDisabled': False,
       'notChargingOther': True,
       'activityTimeGainMs': 0,
       'charging': False,
       'interpolated': None},
    """

    d = safe_api_call(api.get_device_solar_data, get_device()['deviceId'], dates[-3], dates[-1])[1]['solarDailyDataDTOs']
    print(d)

### Get Activities Data

In [66]:
def get_activities_data(dates):
    """
    - ownerFullName
    - ownerProfileImageUrlMedium

    activityId
    activityName
    activityType: {
        typeId
        typeKey
    }
    locationName (hiking?)
    startTimeLocal
    startTimeGMT
    endTimeGMT
    beginTimestamp

    distance
    duration
    elapsedDuration
    movingDuration
    - elevationGain (hiking)
    - elevationLoss (hiking)
    averageSpeed
    maxSpeed
    hasPolyline
    hasImages
    ownerId
    ownerFullName
    ownerProfileImageUrlMedium
    calories
    bmrCalories
    averageHR
    maxHR
    steps
    aerobicTrainingEffect
    anaerobicTrainingEffect
    summarizedExerciseSets: [
        category
        reps
        volume
        duration
        sets
        maxWeight
    ]

    lapCount
    totalSets
    activeSets
    totalReps

    activityTrainingLoad
    minActivityLapDuration

    moderateIntensityMinutes
    vigorousIntensityMinutes

    hrTimeInZone_1
    hrTimeInZone_2
    hrTimeInZone_3
    hrTimeInZone_4
    hrTimeInZone_5

    pr
    """

    result = []

    keys_to_get = [
        'activityId',
        'activityName',
        # activityType: {
        #     typeId
        #     typeKey
        # }

        # 'ownerId',
        # 'ownerFullName',
        # 'ownerProfileImageUrlMedium',
        # 'hasImages',

        # 'locationName',  # (hiking?)
        # 'elevationGain',  # (hiking)
        # 'elevationLoss',  # (hiking)

        'startTimeLocal',
        'startTimeGMT',
        'endTimeGMT',
        'beginTimestamp',

        'pr',

        'duration',
        'elapsedDuration',
        'movingDuration',

        'calories',
        'bmrCalories',

        'steps',
        'distance',

        'averageSpeed',
        'maxSpeed',

        'averageHR',
        'maxHR',
        'hrTimeInZone_1',
        'hrTimeInZone_2',
        'hrTimeInZone_3',
        'hrTimeInZone_4',
        'hrTimeInZone_5',

        'lapCount',
        'totalSets',
        'activeSets',
        'totalReps',

        'aerobicTrainingEffect',
        'anaerobicTrainingEffect',
        'moderateIntensityMinutes',
        'vigorousIntensityMinutes',
        'activityTrainingLoad'
    ]
    activities = safe_api_call(api.get_activities_by_date, dates[0], dates[-1])[1]
    for activity in activities:
        activity_data = {}
        for key in keys_to_get:
            activity_data[key] = activity.get(key, None)
            if key == 'activityName':
                activity_data['activityType'] = activity['activityType'].get('typeKey', 'Unknown').title().replace('_', ' ')

            # Get total calories
            if key == 'bmrCalories':
                activity_data['totalCalories'] = activity['bmrCalories'] + activity['calories']

        result.append(activity_data)
    return result

### Get Health Data

In [6]:
def get_health_data(date):
    """
    """

    to_pop = [
        'userProfileId',
        'userDailySummaryId',
        'burnedKilocalories',
        'wellnessActiveKilocalories',
        'netRemainingKilocalories',
        'rule',
        'wellnessStartTimeGmt',
        'wellnessStartTimeLocal',
        'wellnessEndTimeGmt',
        'wellnessEndTimeLocal',
        'durationInMilliseconds',
        'wellnessDescription',
        'includesWellnessData',
        'includesActivityData',
        'includesCalorieConsumedData',
        'privacyProtected',
        'floorsAscended',
        'floorsDescended',
        'lastSevenDaysAvgRestingHeartRate',
        'source',
        'lastSyncTimestampGMT',
        'bodyBatteryMostRecentValue',
        'bodyBatteryVersion',
        # 'averageSpo2',
        'lowestSpO2Value',
        'highestSpO2Value',
        'lowestSpo2',
        'latestSpo2',
        'latestSpo2ReadingTimeGmt',
        'latestSpo2ReadingTimeLocal',
        'latestSpo2ReadingTimeLocalaverageMonitoringEnvironmentAltitude',
        'restingCaloriesFromActivity',
        'latestRespirationValue',
        'latestRespirationTimeGMT',
        'respirationAlgorithmVersion',
        'ageGroup',
        'averageMonitoringEnvironmentAltitude',
        # 'bodyBatteryChargedValue',
        # 'bodyBatteryDrainedValue',
        # 'bodyBatteryHighestValue',
        # 'bodyBatteryLowestValue',
        # 'bodyBatteryDuringSleep',
        'wellnessKilocalories',
        'consumedKilocalories',
        'remainingKilocalories',
        'netCalorieGoal',
        'wellnessDistanceMeters',
        'userNote',
        'sleepingSeconds',
        'minAvgHeartRate',
        'maxAvgHeartRate',
        'abnormalHeartRateAlertsCount',
        'unmeasurableSleepSeconds',
        # 'measurableAsleepDuration',
        # 'measurableAwakeDuration',
        'stressPercentage',
        'restStressPercentage',
        'activityStressPercentage',
        'uncategorizedStressPercentage',
        'lowStressPercentage',
        'mediumStressPercentage',
        'highStressPercentage',
        'restStressDuration',
        'userFloorsAscendedGoal',
    ]

    health_data = safe_api_call(api.get_stats, date)[1]
    health_data['fitnessAge'] = int(safe_api_call(api.get_fitnessage_data, date)[1]['fitnessAge'])

    # Heart rate data
    hdata = safe_api_call(api.get_heart_rates, date)[1]['heartRateValues']
    if hdata:
        heart_rates = [v[1] for v in hdata if v[1] is not None]
        health_data['avgHeartRate'] = float(np.array(heart_rates).mean())

    # Get sleep data
    health_data.update(get_sleep_data(date))

    # Get hydration data
    health_data.update(get_hydration_data(date))

    for key in to_pop:
        try:
            health_data.pop(key)
        except KeyError as e:
            pass

    # Convert/Add some columns
    convert_dict = {}
    for key, value in health_data.items():
        if value:
            if 'Meters' in key:
                convert_dict[key.replace('Meters', 'Miles')] = value / 1609
            elif 'Seconds' in key:
                convert_dict[key.replace('Seconds', 'Hours')] = value / 3600
            elif 'Duration' in key:
                convert_dict[key.replace('Duration', 'Hours')] = value / 3600
            elif 'Minutes' in key:
                convert_dict[key.replace('Minutes', 'Hours')] = value / 60
            elif 'InML' in key:
                convert_dict[key.replace('InML', 'InCups')] = value / 240

    # Update
    health_data.update(convert_dict)

    return health_data

In [ ]:
def get_healths_data(dates=None):
    data = []
    for date in tqdm(dates or get_date_range()):
        data.append(get_health_data(date))
    return data

## 📆 Get Date Range

In [8]:
dates = get_date_range(datetime.datetime.strptime('12-31-2025', '%m-%d-%Y'))

['2025-01-01',
 '2025-01-02',
 '2025-01-03',
 '2025-01-04',
 '2025-01-05',
 '2025-01-06',
 '2025-01-07',
 '2025-01-08',
 '2025-01-09',
 '2025-01-10',
 '2025-01-11',
 '2025-01-12',
 '2025-01-13',
 '2025-01-14',
 '2025-01-15',
 '2025-01-16',
 '2025-01-17',
 '2025-01-18',
 '2025-01-19',
 '2025-01-20',
 '2025-01-21',
 '2025-01-22',
 '2025-01-23',
 '2025-01-24',
 '2025-01-25',
 '2025-01-26',
 '2025-01-27',
 '2025-01-28',
 '2025-01-29',
 '2025-01-30',
 '2025-01-31',
 '2025-02-01',
 '2025-02-02',
 '2025-02-03',
 '2025-02-04',
 '2025-02-05',
 '2025-02-06',
 '2025-02-07',
 '2025-02-08',
 '2025-02-09',
 '2025-02-10',
 '2025-02-11',
 '2025-02-12',
 '2025-02-13',
 '2025-02-14',
 '2025-02-15',
 '2025-02-16',
 '2025-02-17',
 '2025-02-18',
 '2025-02-19',
 '2025-02-20',
 '2025-02-21',
 '2025-02-22',
 '2025-02-23',
 '2025-02-24',
 '2025-02-25',
 '2025-02-26',
 '2025-02-27',
 '2025-02-28',
 '2025-03-01',
 '2025-03-02',
 '2025-03-03',
 '2025-03-04',
 '2025-03-05',
 '2025-03-06',
 '2025-03-07',
 '2025-03-

## ⚕️ Build Health Data
### Create the Health Dataframe
No longer used because we're going to just be updating the sheet now.

In [14]:
# health_data = get_healths_data(dates)
# df = pd.DataFrame(health_data).convert_dtypes()

### Update the Health Dataset
Read current data and just append rows of new health data

In [166]:
df = lib.read_data(project.raw_file)

PortfolioLogger.lib.tools: INFO: Encoding: ascii
PortfolioLogger.lib.tools: INFO: Stripping whitespaces from raw_data.csv
PortfolioLogger.lib.performance: INFO: <function read_data at 0x10cb311c0> took 0.075 secs to complete.


In [167]:
df = df.drop('Unnamed: 0', axis=1)  # I don't know why it keeps adding this column...

### Get dates to update

In [168]:
dates_to_update = sorted(list(set(dates).difference(set(df['Date'].tolist()))))
dates_to_update

['2025-12-31']

In [169]:
health_data = get_healths_data(dates_to_update)

100%|██████████| 1/1 [00:00<00:00,  1.82it/s]


### Fix the columns names and order

In [170]:
remap_columns = {
    'uuid': 'uuid',
    'calendarDate': 'Date',

    'fitnessAge': 'Fitness Age',

    # Calories
    'totalKilocalories': 'Calories',
    'activeKilocalories': 'Active Calories',
    'bmrKilocalories': 'Resting Calories',

    # Hydration
    'hydrationValueInML': 'Hydration Value In ML',
    'hydrationGoalInML': 'Hydration Goal In ML',
    'sweatLossInML': 'Sweat Loss In ML',
    'hydrationValueInCups': 'Hydration Value In Cups',
    'hydrationGoalInCups': 'Hydration Goal In Cups',
    'sweatLossInCups': 'Sweat Loss In Cups',

    # Heart Rate
    'avgHeartRate': 'Average Heart Rate',
    'minHeartRate': 'Min Heart Rate',
    'maxHeartRate': 'Max Heart Rate',
    'restingHeartRate': 'Resting Heart Rate',
    'hrvStatus': 'Heart Rate Variability Qualifier',

    # Peripheral Oxyge Saturation
    'averageSpo2': 'Average Sp 02',

    # Respiration
    'avgWakingRespirationValue': 'Avg Waking Respiration Value',
    'highestRespirationValue': 'Highest Respiration Value',
    'lowestRespirationValue': 'Lowest Respiration Value',

    # Steps/Distance
    'totalSteps': 'Total Steps',
    'totalDistanceMeters': 'Total Distance Meters',
    'totalDistanceMiles': 'Total Distance Miles',
    'dailyStepGoal': 'Daily Step Goal',

    # Floors
    'floorsAscendedInMeters': 'Floors Ascended In Meters',
    'floorsDescendedInMeters': 'Floors Descended In Meters',

    'floorsAscendedInMiles': 'Floors Ascended In Miles',
    'floorsDescendedInMiles': 'Floors Descended In Miles',

    # Activity
    'activeSeconds': 'Active Seconds',
    'highlyActiveSeconds': 'Highly Active Seconds',
    'sedentarySeconds': 'Sedentary Seconds',
    'moderateIntensityMinutes': 'Moderate Intensity Minutes',
    'vigorousIntensityMinutes': 'Vigorous Intensity Minutes',
    'intensityMinutesGoal': 'Intensity Minutes Goal',

    'activeHours': 'Active Hours',
    'highlyActiveHours': 'Highly Active Hours',
    'sedentaryHours': 'Sedentary Hours',
    'moderateIntensityHours': 'Moderate Intensity Hours',
    'vigorousIntensityHours': 'Vigorous Intensity Hours',
    'intensityHoursGoal': 'Intensity Hours Goal',

    # Stress
    'averageStressLevel': 'Average Stress Level',
    'totalStressDuration': 'Total Stress Seconds',
    'stressDuration': 'Stress Seconds',
    'maxStressLevel': 'Max Stress Level',
    'uncategorizedStressDuration': 'Uncategorized Stress Seconds',
    'lowStressDuration': 'Low Stress Seconds',
    'mediumStressDuration': 'Medium Stress Seconds',
    'highStressDuration': 'High Stress Seconds',
    'activityStressDuration': 'Activity Stress Seconds',
    'stressQualifier': 'Stress Qualifier',

    'stressHours': 'Stress Hours',
    'activityStressHours': 'Activity Stress Hours',
    'uncategorizedStressHours': 'Uncategorized Stress Hours',
    'totalStressHours': 'Total Stress Hours',
    'lowStressHours': 'Low Stress Hours',
    'mediumStressHours': 'Medium Stress Hours',
    'highStressHours': 'High Stress Hours',

    # Body battery
    'bodyBatteryAtWakeTime': 'Body Battery',
    'bodyBatteryChargedValue': 'Body Battery Charged Value',
    'bodyBatteryDrainedValue': 'Body Battery Drained Value',
    'bodyBatteryHighestValue': 'Body Battery Highest Value',
    'bodyBatteryLowestValue': 'Body Battery Lowest Value',
    'bodyBatteryDuringSleep': 'Body Battery During Sleep',

    # Sleep data
    'sleepStartTimestampGMT': 'Sleep Start Timestamp GMT',
    'sleepEndTimestampGMT': 'Sleep End Timestamp GMT',
    'sleepStartTimestampLocal': 'Sleep Start Timestamp Local',
    'sleepEndTimestampLocal': 'Sleep End Timestamp Local',

    'sleepTimeSeconds': 'Sleep Time Seconds',
    'sleepScore': 'Sleep Score',
    'sleepScoreQuality': 'Sleep Quality',
    'sleepScoreFeedback': 'Sleep Feedback',

    'measurableAsleepDuration': 'Measurable Asleep Seconds',
    'measurableAwakeDuration': 'Measurable Awake Seconds',

    'lightSleepSeconds': 'Light Sleep Seconds',
    'deepSleepSeconds': 'Deep Sleep Seconds',
    'remSleepSeconds': 'Rem Sleep Seconds',
    'awakeSleepSeconds': 'Awake Sleep Seconds',

    'sleepTimeHours': 'Sleep Time Hours',
    'measurableAsleepHours': 'Measurable Asleep Hours',
    'measurableAwakeHours': 'Measurable Awake Hours',
    'deepSleepHours': 'Deep Sleep Hours',
    'lightSleepHours': 'Light Sleep Hours',
    'remSleepHours': 'Rem Sleep Hours',
    'awakeSleepHours': 'Awake Sleep Hours',

    'averageRespirationValue': 'Average Respiration Value',
    'awakeCount': 'Awake Count',
    'avgSleepStress': 'Avg Sleep Stress',

    'stressSleepQuality': 'Stress Sleep Quality',
    'awakeCountQuality': 'Awake Count Quality',
    'remSleepQuality': 'Rem Sleep Quality',
    'restlessnessSleepQuality': 'Restlessness Sleep Quality',
    'lightSleepQuality': 'Light Sleep Quality',
    'deepSleepQuality': 'Deep Sleep Quality',

    'avgOvernightHrv': 'Average Overnight Hrv',
}

In [171]:
health_data_fixed = []
for data in health_data:
    data_fixed ={}
    for k, v in data.items():
        renamed_column = remap_columns.get(k)
        if renamed_column:
            data_fixed[renamed_column] = v
        else:
            log.error(f'No column found for {k}')

    health_data_fixed.append(data_fixed)
for data in health_data_fixed:
    df.loc[len(df)] = data
df.tail()

PortfolioLogger.Personal Health Analysis: ERROR: No column found for averageSpO2Value
PortfolioLogger.Personal Health Analysis: ERROR: No column found for averageSpO2HRSleep
/var/folders/_9/nyjys43x4nv6dxyk3y75vzh80000gn/T/ipykernel_1502/800436574.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df.loc[len(df)] = data


,uuid,Date,Fitness Age,Calories,Active Calories,Resting Calories,Hydration Value In ML,Hydration Goal In ML,Sweat Loss In ML,Hydration Value In Cups,...,Average Respiration Value,Awake Count,Avg Sleep Stress,Stress Sleep Quality,Awake Count Quality,Rem Sleep Quality,Restlessness Sleep Quality,Light Sleep Quality,Deep Sleep Quality,Average Overnight Hrv
360,85c2fbd9b8ac4c4885ac252a849e712c,2025-12-27,30,1653.0,136.0,1517.0,2365.880,2839.056,NaN,9.857833,...,16.0,2.0,26.0,Poor,Fair,Fair,Fair,Fair,Fair,43.0
361,85d2432b048c4c0dafd5f6dc41dc17e2,2025-12-28,30,3166.0,1649.0,1517.0,2839.056,2839.056,NaN,11.829400,...,15.0,1.0,22.0,Fair,Good,Fair,Excellent,Fair,Excellent,43.0
362,dbaaefc6bb394050b230ad6c022c4c7b,2025-12-29,30,1242.0,392.0,850.0,709.764,2839.056,NaN,2.957350,...,14.0,0.0,23.0,Fair,Excellent,Excellent,Excellent,Excellent,Excellent,40.0
363,8f1223ed49024b40b7e4d94942c3a80c,2025-12-30,30,1040.0,266.0,774.0,NaN,2839.056,NaN,NaN,...,13.0,1.0,18.0,Fair,Good,Good,Good,Fair,Fair,55.0
364,2eec9a682d2948d585a455e57d12dd9f,2025-12-31,30,1766.0,249.0,1517.0,1182.940,2839.056,NaN,4.928917,...,15.0,0.0,9.0,EXCELLENT,EXCELLENT,FAIR,EXCELLENT,FAIR,GOOD,64.0


In [173]:
df1 = df.copy()

In [174]:
title_columns = ['Qualifier', 'Quality', 'Feedback']
for col, series in df1.items():
    for item in title_columns:
        if re.search(item, col, re.IGNORECASE):
            df1[col] = series.str.title().str.replace('_', ' ')
df1.head()

,uuid,Date,Fitness Age,Calories,Active Calories,Resting Calories,Hydration Value In ML,Hydration Goal In ML,Sweat Loss In ML,Hydration Value In Cups,...,Average Respiration Value,Awake Count,Avg Sleep Stress,Stress Sleep Quality,Awake Count Quality,Rem Sleep Quality,Restlessness Sleep Quality,Light Sleep Quality,Deep Sleep Quality,Average Overnight Hrv
0,8f2ddf09f3e04ac780a5470c4099a803,2025-01-01,30,2322.0,833.0,1489.0,2880.000,2937.056,98.0,12.000000,...,13.0,2.0,11.0,Excellent,Fair,Excellent,Fair,Fair,Fair,NaN
1,4ab88cfdf283481fa593979341aa8332,2025-01-02,30,1960.0,471.0,1489.0,946.353,2839.056,NaN,3.943138,...,13.0,0.0,35.0,Poor,Excellent,Fair,Excellent,Excellent,Excellent,NaN
2,a6433536e16a492e8a7655909869ea9c,2025-01-03,30,1878.0,389.0,1489.0,946.353,3052.056,213.0,3.943138,...,13.0,3.0,14.0,Good,Fair,Excellent,Fair,Good,Fair,NaN
3,8709a878c8b8459f8769c1ff58d886d3,2025-01-04,30,2388.0,899.0,1489.0,3785.410,2839.056,NaN,15.772542,...,13.0,2.0,17.0,Fair,Fair,Fair,Fair,Fair,Excellent,NaN
4,e929e59be1114b58a17d010d389233ed,2025-01-05,30,2536.0,1047.0,1489.0,3785.410,2839.056,NaN,15.772542,...,14.0,4.0,41.0,Poor,Poor,Poor,Poor,Poor,Fair,NaN


### Convert some types

In [175]:
df1['Date'] = pd.to_datetime(df1['Date'])
for col in ['Sleep Start Timestamp GMT', 'Sleep End Timestamp GMT', 'Sleep Start Timestamp Local', 'Sleep End Timestamp Local']:
    df1[col] = pd.to_datetime(df1[col])

### Export the data

In [176]:
df1.to_csv(project.raw_file)

## 🏋🏽‍♂️ Create the Activities Dataframe

In [100]:
activities_data = get_activities_data(dates)
df = pd.DataFrame(activities_data).convert_dtypes()
df['startTimeLocal'] = pd.to_datetime(df['startTimeLocal'])

### Get end time from startTimeLocal and duration

In [ ]:
df['endTimeLocal'] = df.apply(lambda row: row['startTimeLocal'] + datetime.timedelta(seconds=row['duration']), axis=1)
df['endTimeLocal'] = pd.to_datetime(df['endTimeLocal'])

In [115]:
remap_columns = {
    'activityId': 'id',
    'activityName': 'Activity Name',
    'activityType': 'Activity Type',

    'startTimeLocal': 'Start Time',
    'endTimeLocal': 'End Time',

    'pr': 'Personal Record',

    'duration': 'Duration',

    'calories': 'Active Calories',
    'bmrCalories': 'Resting Calories',
    'totalCalories': 'Total Calories',

    'steps': 'Steps',
    'distance': 'Distance',

    'averageSpeed': 'Average Speed',
    'maxSpeed': 'Max Speed',

    'averageHR': 'Average Heart Rate',
    'maxHR': 'Max Heart Rate',
    'hrTimeInZone_1': 'Heart Rate Zone 1 Duration',
    'hrTimeInZone_2': 'Heart Rate Zone 2 Duration',
    'hrTimeInZone_3': 'Heart Rate Zone 3 Duration',
    'hrTimeInZone_4': 'Heart Rate Zone 4 Duration',
    'hrTimeInZone_5': 'Heart Rate Zone 5 Duration',

    'lapCount': 'Lap Count',
    'totalSets': 'Total Sets',
    'activeSets': 'Active Sets',
    'totalReps': 'Total Reps',

    'aerobicTrainingEffect': 'Aerobic Training Effect',
    'anaerobicTrainingEffect': 'Anaerobic Training Effect',
    'moderateIntensityMinutes': 'Moderate Intensity Minutes',
    'vigorousIntensityMinutes': 'Vigorous Intensity Minutes',
    'activityTrainingLoad': 'Activity Training Load',
}

### Rename and reorder the columns

In [119]:
df1 = df.rename(columns=remap_columns)[remap_columns.values()]

### Fill some null values

In [120]:
df1[['Distance', 'Average Speed', 'Max Speed']] = df1[['Distance', 'Average Speed', 'Max Speed']].fillna(0)

In [121]:
df1

,id,Activity Name,Activity Type,Start Time,End Time,Personal Record,Duration,Active Calories,Resting Calories,Total Calories,...,Heart Rate Zone 5 Duration,Lap Count,Total Sets,Active Sets,Total Reps,Aerobic Training Effect,Anaerobic Training Effect,Moderate Intensity Minutes,Vigorous Intensity Minutes,Activity Training Load
0,19797057456,Hood River County Hiking,Hiking,2025-07-20 06:08:56,2025-07-20 11:02:40.724609,False,17624.724609,1256,352,1608,...,0.0,1,<NA>,<NA>,<NA>,2.6,0.3,183,30,48.557602
1,18850062748,HIIT,Hiit,2025-04-16 17:36:24,2025-04-16 18:19:05.699951,False,2561.699951,364,50,414,...,265.015,1,1,1,6,4.0,4.4,4,38,271.572205
2,18818705880,Yoga,Yoga,2025-04-13 09:33:17,2025-04-13 10:05:03.939941,False,1906.939941,162,38,200,...,0.0,1,1,1,0,1.9,0.5,26,5,21.576126
3,18740874417,HIIT,Hiit,2025-04-05 06:23:40,2025-04-05 07:13:06.037109,False,2966.037109,353,58,411,...,66.998,1,1,1,83,3.4,3.0,12,36,133.019272
4,18688383438,Yoga,Yoga,2025-03-31 04:32:50,2025-03-31 04:40:42.438995,False,472.438995,31,9,40,...,0.0,1,1,1,0,0.4,0.0,7,0,3.030548
5,18682395828,Yoga,Yoga,2025-03-30 08:58:48,2025-03-30 09:09:10.091980,False,622.09198,21,12,33,...,0.0,1,1,1,0,0.2,0.0,3,0,1.818832
6,18670755423,Yoga,Yoga,2025-03-29 06:38:26,2025-03-29 06:44:49.437988,False,383.437988,28,8,36,...,0.0,1,1,1,0,0.4,0.0,4,0,3.312729
7,18661169456,Yoga,Yoga,2025-03-28 05:43:40,2025-03-28 05:54:03.635986,False,623.635986,42,12,54,...,0.0,1,1,1,0,0.6,0.0,9,0,4.36792
8,18651905872,Yoga,Yoga,2025-03-27 05:41:12,2025-03-27 05:49:15.891998,False,483.891998,32,10,42,...,0.0,1,1,1,0,0.4,0.0,7,0,3.002701
9,18647913808,HIIT,Hiit,2025-03-26 16:21:42,2025-03-26 16:52:57.925049,False,1875.925049,264,37,301,...,210.643,1,1,1,53,4.2,3.9,3,27,211.518539


### Export the data

In [122]:
df1.to_csv(project / 'activities_data.csv')